In [21]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import ndtr, expit, ive
from scipy.integrate import quad

# ============================================================
# Market data
# ============================================================

r = 0.05
S0_market = 394.118

# 3M ATM call implied volatility
T_call = 0.25
sigma_black = 0.04635


# Dividend futures data:
# reference intervals [0.25 + k, 1.25 + k], k = 0,...,9
div_futures_market = np.array([
    17.1981,
    18.7223,
    19.4360,
    19.7445,
    19.8833,
    19.9623,
    19.9703,
    20.0066,
    20.0019,
    20.0031,
])

T0_list = np.array([0.25 + k for k in range(10)])
T1_list = np.array([1.25 + k for k in range(10)])


# ============================================================
# Parameter transformation
# ============================================================
# We optimize over unconstrained variables x.
# This guarantees:
# lambda > 0, C_star > 0, C0 > 0,
# and phi^2 < 2 lambda C_star.

def unpack_parameters(x):
    """
    Convert unconstrained optimization variables into model parameters.

    x[0] -> lambda
    x[1] -> C_star
    x[2] -> C0
    x[3] -> phi through a sigmoid to impose the Feller condition
    """
    lam = np.exp(x[0])
    c_star = np.exp(x[1])
    c0 = np.exp(x[2])

    # sigmoid in (0,1)
    z = expit(x[3])

    # Feller constraint: phi^2 < 2 lambda C_star
    phi = np.sqrt(2.0 * lam * c_star) * z

    return lam, c_star, c0, phi


def pack_parameters(lam, c_star, c0, phi):
    """
    Convert model parameters into unconstrained variables.
    Useful for constructing initial guesses.
    """
    x0 = np.log(lam)
    x1 = np.log(c_star)
    x2 = np.log(c0)

    z = phi / np.sqrt(2.0 * lam * c_star)
    z = np.clip(z, 1e-8, 1.0 - 1e-8)
    x3 = np.log(z / (1.0 - z))

    return np.array([x0, x1, x2, x3])


# ============================================================
# Model-implied index level
# ============================================================

def model_index_level(lam, c_star, c0):
    """
    Under the equivalent conditions of Q4:
        S0 = C*/r + (C0 - C*)/(r + lambda)
    """
    return c_star / r + (c0 - c_star) / (r + lam)


# ============================================================
# Model-implied index futures price
# ============================================================

def model_index_futures(T, lam, c_star, c0):
    """
    Q6 formula:
        f_0^S(T) = S0 + (1 - exp(-lambda T))/(r + lambda) * (C* - C0)
    """
    S0_model = model_index_level(lam, c_star, c0)

    return (
        S0_model
        + (1.0 - np.exp(-lam * T)) / (r + lam) * (c_star - c0)
    )


# ============================================================
# Model-implied dividend futures prices
# ============================================================

def model_dividend_future(T0, T1, lam, c_star, c0):
    """
    Q7 formula at t = 0, with T0 > 0:
        f_0^{div}(T0,T1)
        = C* (T1 - T0)
          + (C0 - C*)/lambda * (exp(-lambda T0) - exp(-lambda T1))
    """
    return (
        c_star * (T1 - T0)
        + (c0 - c_star) / lam * (np.exp(-lam * T0) - np.exp(-lam * T1))
    )


def model_all_dividend_futures(lam, c_star, c0):
    return np.array([
        model_dividend_future(T0, T1, lam, c_star, c0)
        for T0, T1 in zip(T0_list, T1_list)
    ])


# ============================================================
# Black call price from a forward
# ============================================================

def black_call_from_forward(F, K, T, sigma):
    """
    Black formula:
        C = exp(-rT) [F Phi(d+) - K Phi(d-)]
    """
    if T <= 0:
        return max(F - K, 0.0)

    if sigma <= 0:
        return np.exp(-r * T) * max(F - K, 0.0)

    vol = sigma * np.sqrt(T)

    d_plus = (np.log(F / K) + 0.5 * sigma ** 2 * T) / vol
    d_minus = d_plus - vol

    return np.exp(-r * T) * (F * ndtr(d_plus) - K * ndtr(d_minus))


# ============================================================
# Model-implied call price using the Bessel density.
# ============================================================

def cir_density_bessel(x, T, lam, c_star, c0, phi):
    """
    Density of C_T under the CIR dynamics, using the Bessel formula from Q11.

    The density is:
        p_C(T,x)
        = a exp(-a(x + exp(-lambda T) C0))
          (exp(lambda T) x / C0)^{q/2}
          I_q(2a sqrt(exp(-lambda T) x C0))

    where:
        q = 2 lambda C* / phi^2 - 1
        a = 2 lambda / (phi^2 (1 - exp(-lambda T)))
    """

    if x <= 0.0:
        return 0.0

    exp_minus = np.exp(-lam * T)

    q = 2.0 * lam * c_star / (phi ** 2) - 1.0
    a = 2.0 * lam / (phi ** 2 * (1.0 - exp_minus))

    z = 2.0 * a * np.sqrt(exp_minus * x * c0)

    # Use the scaled Bessel function:
    # ive(q,z) = exp(-z) I_q(z)
    # so log(I_q(z)) = z + log(ive(q,z)).
    bessel_scaled = ive(q, z)

    if bessel_scaled <= 0.0 or not np.isfinite(bessel_scaled):
        return 0.0

    log_density = (
        np.log(a)
        - a * (x + exp_minus * c0)
        + 0.5 * q * (lam * T + np.log(x) - np.log(c0))
        + z
        + np.log(bessel_scaled)
    )

    return np.exp(log_density)


def model_call_price(T, K, lam, c_star, c0, phi):
    """
    Price a European call on the index using the Bessel density from Q11.

    Under the equivalent conditions of Q4:
        S_T = (C_T + lambda C*/r) / (r + lambda).

    Therefore:
        C_T = (r + lambda) S_T - lambda C*/r.

    The call price is:
        exp(-rT) E[(S_T - K)^+].
    """

    alpha = lam * c_star / r

    # Strike threshold in terms of C_T:
    # S_T > K  <=>  C_T > (r + lambda) K - alpha
    x_strike = (r + lam) * K - alpha

    # C_T is positive, so the integration lower bound is max(x_strike, 0)
    x_lower = max(x_strike, 0.0)

    def integrand(x):
        # Convert C_T = x into S_T
        s = (x + alpha) / (r + lam)

        payoff = max(s - K, 0.0)

        return payoff * cir_density_bessel(
            x=x,
            T=T,
            lam=lam,
            c_star=c_star,
            c0=c0,
            phi=phi,
        )

    integral, error = quad(
        integrand,
        x_lower,
        np.inf,
        epsabs=1e-8,
        epsrel=1e-8,
        limit=200,
    )

    return np.exp(-r * T) * integral


# ============================================================
# Calibration objective
# ============================================================

def objective(x, verbose=False):
    """
    Unweighted sum of squared pricing errors.
    """
    lam, c_star, c0, phi = unpack_parameters(x)

    try:
        # 1. Index level error
        S0_model = model_index_level(lam, c_star, c0)
        err_index = S0_model - S0_market

        # 2. Dividend futures errors
        div_model = model_all_dividend_futures(lam, c_star, c0)
        err_div = div_model - div_futures_market

        # 3. ATM 3M call error
        #
        # We convert the quoted implied volatility into a Black price using
        # the model-implied 3M forward. This is equivalent to matching
        # the market implied volatility under the model forward convention.
        F_call = model_index_futures(T_call, lam, c_star, c0)
        K_call = F_call

        call_market = black_call_from_forward(
            F=F_call,
            K=K_call,
            T=T_call,
            sigma=sigma_black,
        )

        call_model = model_call_price(
            T=T_call,
            K=K_call,
            lam=lam,
            c_star=c_star,
            c0=c0,
            phi=phi,
        )

        err_call = call_model - call_market

        errors = np.concatenate([
            np.array([err_index]),
            err_div,
            np.array([err_call]),
        ])

        sse = float(np.sum(errors ** 2))

        if verbose:
            print("lambda =", lam)
            print("C_star =", c_star)
            print("C0 =", c0)
            print("phi =", phi)
            print("S0 model =", S0_model, "error =", err_index)
            print("Call model =", call_model)
            print("Call market =", call_market, "error =", err_call)
            print("SSE =", sse)

        if not np.isfinite(sse):
            return 1e30

        return sse

    except Exception:
        return 1e30


# ============================================================
# Run calibration
# ============================================================

def run_calibration():
    """
    Run a small multi-start optimization.
    """
    initial_guesses = [
        # lambda, C_star, C0, phi
        (0.80, 20.0, 15.0, 4.0),
        (0.50, 20.0, 15.0, 3.0),
        (1.20, 20.0, 15.0, 4.5),
        (0.80, 21.0, 16.0, 4.0),
        (0.30, 20.0, 14.0, 2.0),
        (1.50, 20.0, 16.0, 5.0),
    ]

    best_result = None

    for guess in initial_guesses:
        x0 = pack_parameters(*guess)

        result = minimize(
            objective,
            x0,
            method="Nelder-Mead",
            options={
                "maxiter": 3000,
                "xatol": 1e-10,
                "fatol": 1e-10,
                "disp": False,
            },
        )

        if best_result is None or result.fun < best_result.fun:
            best_result = result

    return best_result


In [22]:
result = run_calibration()

lam, c_star, c0, phi = unpack_parameters(result.x)

S0_model = model_index_level(lam, c_star, c0)
div_model = model_all_dividend_futures(lam, c_star, c0)

F_call = model_index_futures(T_call, lam, c_star, c0)
K_call = F_call

call_market = black_call_from_forward(
        F=F_call,
        K=K_call,
        T=T_call,
        sigma=sigma_black,
)

call_model = model_call_price(
        T=T_call,
        K=K_call,
        lam=lam,
        c_star=c_star,
        c0=c0,
        phi=phi,
)

print("\n===== Calibration result =====")
print(f"Success: {result.success}")
print(f"Objective SSE: {result.fun:.10g}")

print("\nParameters:")
print(f"lambda  = {lam:.8f}")
print(f"C_star  = {c_star:.8f}")
print(f"C0      = {c0:.8f}")
print(f"phi     = {phi:.8f}")

print("\nFeller condition:")
print(f"phi^2          = {phi ** 2:.8f}")
print(f"2 lambda Cstar = {2.0 * lam * c_star:.8f}")

print("\nIndex level:")
print(f"S0 market = {S0_market:.8f}")
print(f"S0 model  = {S0_model:.8f}")
print(f"error     = {S0_model - S0_market:.8f}")

print("\n3M ATM call:")
print(f"Forward used = {F_call:.8f}")
print(f"Call market  = {call_market:.8f}")
print(f"Call model   = {call_model:.8f}")
print(f"error        = {call_model - call_market:.8f}")





===== Calibration result =====
Success: True
Objective SSE: 0.0008697073011

Parameters:
lambda  = 0.79450625
C_star  = 19.99997095
C0      = 15.03518978
phi     = 4.36322864

Feller condition:
phi^2          = 19.03776415
2 lambda Cstar = 31.78020402

Index level:
S0 market = 394.11800000
S0 model  = 394.12050319
error     = 0.00250319

3M ATM call:
Forward used = 395.17955460
Call market  = 3.60816074
Call model   = 3.60816074
error        = 0.00000000


In [23]:
print("\nDividend futures fit:")
print("k | market    | model     | error")
print("--|-----------|-----------|-----------")

for k, (mkt, mod) in enumerate(zip(div_futures_market, div_model)):
    print(f"{k:1d} | {mkt:9.4f} | {mod:9.4f} | {mod - mkt:9.4f}")


Dividend futures fit:
k | market    | model     | error
--|-----------|-----------|-----------
0 |   17.1981 |   17.1915 |   -0.0066
1 |   18.7223 |   18.7311 |    0.0088
2 |   19.4360 |   19.4267 |   -0.0093
3 |   19.7445 |   19.7410 |   -0.0035
4 |   19.8833 |   19.8829 |   -0.0004
5 |   19.9623 |   19.9471 |   -0.0152
6 |   19.9703 |   19.9761 |    0.0058
7 |   20.0066 |   19.9892 |   -0.0174
8 |   20.0019 |   19.9951 |   -0.0068
9 |   20.0031 |   19.9978 |   -0.0053


In [24]:
# ============================================================
# Q13: Initial hedge ratio using a 2Y index futures
# ============================================================

def q13_initial_futures_position(lam, c_star, c0, phi):
    """
    Compute the initial number of 2Y index futures contracts needed to hedge
    a 1Y ATM-forward call on the Euro Stoxx 50.

    Hedge ratio:
        n0 = (d Call_0 / d C0) / (d Futures_0 / d C0)
    """

    T_option = 1.0
    T_futures = 2.0

    # ATM-forward convention for the 1Y call
    K = model_index_futures(T_option, lam, c_star, c0)

    # finite-difference step
    h = 1e-4 * max(1.0, c0)

    # Keep the strike fixed when bumping C0
    call_up = model_call_price(
        T=T_option,
        K=K,
        lam=lam,
        c_star=c_star,
        c0=c0 + h,
        phi=phi,
    )

    call_down = model_call_price(
        T=T_option,
        K=K,
        lam=lam,
        c_star=c_star,
        c0=c0 - h,
        phi=phi,
    )

    d_call_d_c0 = (call_up - call_down) / (2.0 * h)

    # From Q6:
    # f_0^S(T) = C*/r + exp(-lambda T)/(r+lambda)(C0-C*)
    d_futures_d_c0 = np.exp(-lam * T_futures) / (r + lam)

    n0 = d_call_d_c0 / d_futures_d_c0

    return n0, d_call_d_c0, d_futures_d_c0, K


# After calibration, compute Q13 hedge ratio
n0, d_call_d_c0, d_fut_d_c0, K_q13 = q13_initial_futures_position(
    lam=lam,
    c_star=c_star,
    c0=c0,
    phi=phi,
)

print("\n===== Q13: Initial 2Y futures hedge =====")
print(f"1Y ATM-forward strike K = {K_q13:.8f}")
print(f"d Call_0 / d C0          = {d_call_d_c0:.8f}")
print(f"d Futures_0(2Y) / d C0   = {d_fut_d_c0:.8f}")
print(f"Initial number of 2Y futures contracts n0 = {n0:.8f}")


===== Q13: Initial 2Y futures hedge =====
1Y ATM-forward strike K = 397.34329981
d Call_0 / d C0          = 0.31612172
d Futures_0(2Y) / d C0   = 0.24171175
Initial number of 2Y futures contracts n0 = 1.30784591
